# Raw Layer Data Quality (65 Tables) — STATUS REPORT MODE (WARNING ONLY)

**DQ Check Rules (All WARNING — Status Reporting Only):**
- **Row Count** (WARNING) - Status report on ALL 65 tables (gate always passes)
- **Null Check** (WARNING) - Status report on watermark + confirmed key columns (ALL 65 tables)
- **Uniqueness** (WARNING) - Status report on ONLY 3 tables with confirmed keys

**Gate Behavior**: ✅ ALWAYS PASSES (reports failures but does NOT block promotion)

In [0]:
%pip install great-expectations==0.18.19

dbutils.library.restartPython()

  Obtaining dependency information for great-expectations==0.18.19 from https://files.pythonhosted.org/packages/af/36/84641a27d0ac3de942296d3254d4fb3708c03cc0fe58cce86188533b6a97/great_expectations-0.18.19-py3-none-any.whl.metadata
  Using cached great_expectations-0.18.19-py3-none-any.whl.metadata (8.8 kB)
  Obtaining dependency information for altair<5.0.0,>=4.2.1 from https://files.pythonhosted.org/packages/18/62/47452306e84d4d2e67f9c559380aeb230f5e6ca84fafb428dd36b96a99ba/altair-4.2.2-py3-none-any.whl.metadata
  Using cached altair-4.2.2-py3-none-any.whl.metadata (13 kB)
  Obtaining dependency information for colorama>=0.4.3 from https://files.pythonhosted.org/packages/d1/d6/3965ed04c63042e047cb6a3e6ed1a63a35087b6a609aa3a15ed8ac56c221/colorama-0.4.6-py2.py3-none-any.whl.metadata
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Obtaining dependency information for jinja2>=2.10 from https://files.pythonhosted.org/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e5

## 1. Config

In [0]:
# ── Config ───────────────────────────────────────────────────────────────────
CATALOG   = 'mars_dev'
GX_ROOT   = 'dbfs:/cubic_mars/great_expectations_raw'
AUDIT_TABLE = f'{CATALOG}.audit.dq_raw_audit'

# S3 Base Path
S3_BASE = 's3://cubic-mars-pm-s3-datalake-dev-raw-170202974600/chicago_ventra'

# Oracle connection
ORACLE_HOST = '10.3.10.30'
ORACLE_PORT = 1521
ORACLE_USER = dbutils.secrets.get("cubic", "ods_user")
ORACLE_PWD = dbutils.secrets.get("cubic", "ods_pwd")
ORACLE_SERVICE = dbutils.secrets.get("cubic", "ods_service")
JDBC_URL = f'jdbc:oracle:thin:@//{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}'

# Thresholds
ROW_COUNT_EXACT_MATCH = True  # True = exact match, False = ±5% variance
ROW_COUNT_VARIANCE_PCT = 5.0  # (not used when EXACT_MATCH=True, kept for backward compat)
DEFAULT_ROW_FLOOR = 1
SAMPLE_FRACTION = 1.0
print('✅ Config ready - STATUS REPORT MODE (All checks WARNING - gate always passes)')
print(f'✅ Row Count Mode: {"EXACT MATCH (raw == oracle)" if ROW_COUNT_EXACT_MATCH else "VARIANCE (±5%)"}')

# Tables with CONFIRMED keys — uniqueness check ONLY on these 3 tables (WARNING for status reporting)
TABLES_WITH_CONFIRMED_KEYS = {
    'AVAILABILITY_EVENTS',        # STRONG confidence — check AVAILABILITY_EVENT_ID
    'DEVICE_CURRENT_HW_CONFIG',   # CONFIRMED confidence — check DEVICE_KEY, COMPONENT_SERIAL_NBR
    'DEVICE_EVENT'                # CONFIRMED confidence — check DW_DEVICE_EVENT_ID
}

print(' Config ready - STATUS REPORT MODE (All checks WARNING - gate always passes)')

✅ Config ready - STATUS REPORT MODE (All checks WARNING - gate always passes)


## 2. Bootstrap + hardened helpers

In [0]:
import great_expectations as gx
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

def now_utc(): 
    return datetime.now(timezone.utc)

context = gx.get_context(context_root_dir=GX_ROOT)
try:    
    ds = context.sources.add_spark('mars_spark_raw')
except Exception: 
    ds = context.datasources['mars_spark_raw']

RESULTS = []

def get_validator(df, suite_name):
    if 0.0 < SAMPLE_FRACTION < 1.0:
        df = df.sample(False, SAMPLE_FRACTION, seed=42)
    try:    
        asset = ds.add_dataframe_asset(name=suite_name)
    except Exception: 
        asset = ds.get_asset(suite_name)
    br = asset.build_batch_request(dataframe=df)
    context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    v = context.get_validator(batch_request=br, expectation_suite_name=suite_name)
    return v, br

def safe_expect(validator, cols, method, *, severity='warning', column=None, **kw):
    """Add an expectation ONLY if its column exists; else record a skip."""
    if column is not None and column not in cols:
        return False
    meta = kw.pop('meta', {}) or {}
    meta['severity'] = severity
    fn = getattr(validator, method)
    if column is None: 
        fn(meta=meta, **kw)
    else:              
        fn(column, meta=meta, **kw)
    return True

spark.sql(f"""CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
  table_name STRING, schema_name STRING, dq_check STRING, column_name STRING, 
  success BOOLEAN, severity STRING, details STRING, run_ts TIMESTAMP)
  USING DELTA COMMENT 'DQ raw audit - status report, one row per check result'""")

def run_and_score(suite_name, table_name, schema_name, br):
    """Run validation and score results"""
    cp = context.add_or_update_checkpoint(name=suite_name.replace('.','_')+'_cp',
        validations=[{'batch_request': br, 'expectation_suite_name': suite_name}])
    res = cp.run(runtime_configuration={'catch_exceptions': True, 'result_format': 'SUMMARY'})
    vr  = res.list_validation_results()[0]
    ts = now_utc()
    warning_count = 0
    detail = []
    for r in vr['results']:
        cfg = r['expectation_config']
        meta = cfg.get('meta', {}) or {}
        sev = meta.get('severity', 'warning')
        etype = cfg['expectation_type']
        col = cfg['kwargs'].get('column')
        success = bool(r['success'])
        unexp = (r.get('result') or {}).get('unexpected_percent')
        obs   = (r.get('result') or {}).get('observed_value')
        if not success:
            warning_count += 1
        detail.append((table_name, schema_name, etype, col, success, sev,
                      f'{etype}({col}): {obs}' if col else str(obs)[:200], ts))
    st = vr['statistics']
    evaluated = int(st['evaluated_expectations'])
    successful = int(st['successful_expectations'])
    try:
        if detail:
            spark.createDataFrame(detail,
                'table_name string, schema_name string, dq_check string, column_name string, success boolean, severity string, details string, run_ts timestamp'
                ).write.mode('append').saveAsTable(AUDIT_TABLE)
    except Exception as e:
        print(f'    WARN: audit write failed for {suite_name}: {e}')
    
    status = '[PASS]' if successful == evaluated else '[WARN]'
    print(f"{status} {suite_name}: {successful}/{evaluated} ok ({successful/evaluated*100:.0f}%) warnings={warning_count}")
    out = {'suite':suite_name,'table':table_name,'schema':schema_name,'evaluated':evaluated,
           'successful':successful,'warning_count':warning_count,'run_ts':ts}
    RESULTS.append(out)
    return out

print(' helpers ready')

    - No action was taken.
  warnings.warn(message)

    - No action was taken.
  warnings.warn(message)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpeoiruiz_' for ephemeral docs site


✅ helpers ready


## 3. Hardcoded 65 Tables Configuration

In [0]:
# ── All 65 raw tables with NULL check columns (watermark + confirmed key) ──
RAW_TABLES_CONFIG = {

    # ============================================================
    # EDW Schema - 31 tables
    # ============================================================

    'ABP_TAP': {
        'schema': 'edw',
        'null_check_cols': ['TRANSACTION_DTM'],
        'uniqueness_cols': []
    },

    'ACTIVITY_CODE_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'AJ_METRIC_FACT': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'AVAILABILITY_EVENTS': {
        'schema': 'edw',
        'null_check_cols': [
            'TRANSIT_DAY_KEY',
            'AVAILABILITY_EVENT_ID'
        ],
        'uniqueness_cols': ['AVAILABILITY_EVENT_ID']
    },

    'AVAILABILITY_PERIODS': {
        'schema': 'edw',
        'null_check_cols': ['START_DTM'],
        'uniqueness_cols': []
    },

    'AVAILABILITY_RELIEF': {
        'schema': 'edw',
        'null_check_cols': ['START_DTM'],
        'uniqueness_cols': []
    },

    'CASHBOX_EVENT_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'CASHBOX_TYPE_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'DATE_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_BEGIN_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_CURRENT_HW_CONFIG': {
        'schema': 'edw',
        'null_check_cols': ['LAST_REPORTED_DTM'],
        'uniqueness_cols': [
            'DEVICE_KEY',
            'COMPONENT_SERIAL_NBR'
        ]
    },

    'DEVICE_CURRENT_SW_CONFIG': {
        'schema': 'edw',
        'null_check_cols': ['LAST_REPORTED_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_CURRENT_TABLES': {
        'schema': 'edw',
        'null_check_cols': ['MSG_EFFECTIVE_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['INSERTED_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_EVENT': {
        'schema': 'edw',
        'null_check_cols': [
            'EVENT_DAY_KEY',
            'DW_DEVICE_EVENT_ID'
        ],
        'uniqueness_cols': ['DW_DEVICE_EVENT_ID']
    },

    'DEVICE_LAST_SET_EVENT': {
        'schema': 'edw',
        'null_check_cols': ['EVENT_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_LAST_STATE': {
        'schema': 'edw',
        'null_check_cols': ['EVENT_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_LOCATION_HISTORY': {
        'schema': 'edw',
        'null_check_cols': ['CHANGE_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_MESSAGE_TYPE_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'DEVICE_METRIC': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'EVENT_TYPE_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'KPI': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'KPI_DETAIL_EVENTS_BY_DAY': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'KPI_RULES': {
        'schema': 'edw',
        'null_check_cols': ['INSERTED_DTM'],
        'uniqueness_cols': []
    },

    'KPI_SUMMARY_BY_DAY': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'KPI_TARGET': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'LATE_TRANSACTION_DETAIL': {
        'schema': 'edw',
        'null_check_cols': ['POSTING_DAY_KEY'],
        'uniqueness_cols': []
    },

    'METRIC_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'METRIC_SUMMARY_BY_DAY': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'READ_TRANSACTION': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'STOP_POINT_DIMENSION': {
        'schema': 'edw',
        'null_check_cols': ['EDW_UPDATED_DTM'],
        'uniqueness_cols': []
    },

    'USE_TRANSACTION_DAILY': {
        'schema': 'edw',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    


    # ============================================================
    # NCS_STAGE Schema - 22 tables
    # ============================================================

    'ACTIVITY_CODE': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'CASHBOX_EVENT': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'CASHBOX_INVENTORY': {
        'schema': 'ncs_stage',
        'null_check_cols': ['EVENT_DTM'],
        'uniqueness_cols': []
    },

    'CASHBOX_MANUAL_COUNTS': {
        'schema': 'ncs_stage',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'CASHBOX_TRACKING': {
        'schema': 'ncs_stage',
        'null_check_cols': ['EVENT_DTM'],
        'uniqueness_cols': []
    },

    'CASHBOX_TYPE': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'COMPONENT_TYPE': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'DAILY_CASHBOX_DETAIL': {
        'schema': 'ncs_stage',
        'null_check_cols': ['EVENT_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE': {
        'schema': 'ncs_stage',
        'null_check_cols': ['UPDATED_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_CONTROL_GROUP': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'DEVICE_CONTROL_GROUP_TYPE': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'DEVICE_CURRENT_HW_CONFIG': {
        'schema': 'ncs_stage',
        'null_check_cols': ['LAST_REPORTED_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_END_OF_DAY': {
        'schema': 'ncs_stage',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'DEVICE_END_OF_DAY_MSG_COUNT': {
        'schema': 'ncs_stage',
        'null_check_cols': ['TRANSIT_DAY_KEY'],
        'uniqueness_cols': []
    },

    'DEVICE_EVENT_HISTORY': {
        'schema': 'ncs_stage',
        'null_check_cols': ['EVENT_DTM'],
        'uniqueness_cols': []
    },

    'DEVICE_TYPE': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'EVENT': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'SALE_TRANSACTION': {
        'schema': 'ncs_stage',
        'null_check_cols': ['TRANSACTION_DTM'],
        'uniqueness_cols': []
    },

    'SALE_TRANSACTION_DEVICE_MSG': {
        'schema': 'ncs_stage',
        'null_check_cols': ['TRANSACTION_DTM'],
        'uniqueness_cols': []
    },

    'STOP_POINT': {
        'schema': 'ncs_stage',
        'null_check_cols': ['UPDATED_DTM'],
        'uniqueness_cols': []
    },

    'STOP_POINT_TYPE': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'TRANSIT_FACILITY': {
        'schema': 'ncs_stage',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },


    # ============================================================
    # CTA Schema - 12 tables
    # ============================================================

    'ABP_USE_TRAN_TIMING_DATA': {
        'schema': 'cta',
        'null_check_cols': ['DW_INSERTED_DTM'],
        'uniqueness_cols': []
    },

    'AVAILABILITY_LEVELS': {
        'schema': 'cta',
        'null_check_cols': ['INSERTED_DTM'],
        'uniqueness_cols': []
    },

    'KPI_AGENCY_MAP': {
        'schema': 'cta',
        'null_check_cols': ['UPDATED_DTM'],
        'uniqueness_cols': []
    },

    'KPI_MONTHLY_SUMMARY': {
        'schema': 'cta',
        'null_check_cols': ['MONTH_DTM'],
        'uniqueness_cols': []
    },

    'KPI_TVM_DATE_TABLE': {
        'schema': 'cta',
        'null_check_cols': ['_raw_ingest_ts'],
        'uniqueness_cols': []
    },

    'KPI_TVM_TABLE': {
        'schema': 'cta',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'KPI_VARIABLE_FEE': {
        'schema': 'cta',
        'null_check_cols': ['MONTH_DTM'],
        'uniqueness_cols': []
    },

    'METRIX_FACILITY_MAP': {
        'schema': 'cta',
        'null_check_cols': ['_ingest_ts'],
        'uniqueness_cols': []
    },

    'MM_DAILY_TRANSACTION_TIMING': {
        'schema': 'cta',
        'null_check_cols': ['POSTING_DAY_KEY'],
        'uniqueness_cols': []
    },

    'SLDC_MONTHLY_SUMMARY': {
        'schema': 'cta',
        'null_check_cols': ['MONTH_DTM'],
        'uniqueness_cols': []
    },

    'SERVICENOW_AVAILABILITY_EVENTS': {
        'schema': 'cta',
        'null_check_cols': ['SN_U_END_DTM'],
        'uniqueness_cols': []
    },

    'SERVICENOW_DATA_FROM_JUMPBOX': {
        'schema': 'cta',
        'null_check_cols': ['U_START_DTM'],
        'uniqueness_cols': []
    }
}

print(f' Config loaded: {len(RAW_TABLES_CONFIG)} tables')

✅ Config loaded: 64 tables


## 4. Read from S3 and Oracle

In [0]:
# ============================================================================
# UPDATED SECTION 4: Read from S3 and Oracle
# ============================================================================
#
# REPLACE THIS ENTIRE SECTION in your notebook
# Find: "## 4. Read from S3 and Oracle"
# Delete: Everything from there until "## 5. Run validation..."
# Paste: This entire section
#
# ============================================================================

# MAGIC %md
# MAGIC ## 4. Read from S3 and Oracle

def read_from_oracle(schema, table_name):
    """
     IMPROVED: Fetch row count from Oracle source-of-truth
    
    FIXES:
    - Handles custom Oracle types (ARRAY, OBJECT, VARRAY) gracefully
    - Uses COUNT query to avoid reading entire table with custom types
    - No more [UNRECOGNIZED_SQL_TYPE] errors
    - Returns accurate row count (integer, not DataFrame)
    
    CHANGES FROM ORIGINAL:
    - Uses wrapped COUNT query instead of full table read
    - Extracts count value from query result (returns integer)
    - Fetchsize reduced to 1 (only need count)
    - Much faster: 1 second vs. potentially minutes for 1.2B rows
    
    LOGIC IMPACT:
     Same validation logic (no changes to how validation works)
     Same row count bounds calculation (no changes to formulas)
     Just more accurate (actual count instead of fallback 1)
    """
    try:
        full_table = f"{schema}.{table_name}"
        
        # Use COUNT() query wrapped as subquery
        # This avoids reading the actual data (which has custom types)
        count_query = f"(SELECT COUNT(*) as cnt FROM {full_table}) t"
        
        df = spark.read.format('jdbc').options(
            url=JDBC_URL,
            dbtable=count_query,
            user=ORACLE_USER,
            password=ORACLE_PWD,
            driver='oracle.jdbc.driver.OracleDriver',
            fetchsize='1',
            connectionTimeout='30000',
            queryTimeout='120'
        ).load()
        
        # Extract the actual count value from the result
        # df.collect()[0][0] gets: first row [0], first column [0]
        actual_count = df.collect()[0][0]
        return actual_count
        
    except Exception as e:
        print(f'     Oracle read failed for {schema}.{table_name}: {str(e)[:100]}')
        return None

def read_from_s3_raw(schema, table_name):
    """Read from S3 raw layer"""
    s3_path = f'{S3_BASE}/{schema.lower()}/{table_name.lower()}/'
    try:
        df = spark.read.format('parquet').load(s3_path)
        return df
    except Exception as e:
        return None

print(' Read functions ready')

# ============================================================================
# END OF SECTION 4
# ============================================================================

✅ Read functions ready


## 5. Run validation for all 65 tables

In [0]:
# ============================================================================
# SECTION 5: Run validation for all 65 tables
# SUPER ROBUST VERSION - Handles ALL errors gracefully
# ============================================================================
#
# THIS VERSION:
# - Catches Oracle timeouts (skips row count check, continues)
# - Catches S3 path errors (skips table, continues)
# - Catches Decimal conversion errors (sets to None, continues)
# - Catches ANY other errors (logs, skips, continues)
# - NEVER crashes - always moves to next table
#
# Copy this entire code and paste in Section 5
#
# ============================================================================

# MAGIC %md
# MAGIC ## 5. Run validation for all 65 tables

ts = now_utc()
failed_tables = []  # Track tables that fail

for i, (table_name, cfg) in enumerate(sorted(RAW_TABLES_CONFIG.items()), 1):
    try:  # ← OUTER TRY: Catch ANY error for this table
        schema = cfg['schema']
        null_check_cols = cfg['null_check_cols']
        uniqueness_cols = cfg['uniqueness_cols']
        table_upper = table_name.upper()
        has_uniqueness = len(uniqueness_cols) > 0
        marker = ' [UNIQUENESS]' if has_uniqueness else ''
        
        print(f'\n[{i:2d}/65] {schema:12s} . {table_name:35s}{marker}', end='')
        
        # Read S3 data (NEW: Try/except for S3 path errors)
        try:
            raw_df = read_from_s3_raw(schema, table_name)
        except Exception as e:
            error_msg = str(e)[:100]
            print(f' ✗ (S3 error: {error_msg})')
            failed_tables.append((i, table_name, f'S3 read failed: {error_msg}'))
            continue
        
        if raw_df is None: 
            print(' ✗ (S3 not found)')
            failed_tables.append((i, table_name, 'S3 not found'))
            continue
        
        print(f' ✓ {raw_df.count():>10,d} rows', end='')
        
        # Setup validator
        suite = f'raw.{schema}_{table_name}'
        v, br = get_validator(raw_df, suite)
        cols = {f.name for f in raw_df.schema.fields}
        
        # Get Oracle row count (NOW RETURNS INTEGER, NOT DATAFRAME)
        # ✅ Oracle failures are handled gracefully by read_from_oracle
        oracle_count = read_from_oracle(schema, table_name)
        
        # CRITICAL: Convert Decimal to int if not None
        if oracle_count is not None:
            try:
                oracle_count = int(oracle_count)
            except Exception as e:
                print(f'   ⚠️  Cannot convert oracle_count: {str(e)[:50]}')
                oracle_count = None
        
        # CHECK 1: Row Count (WARNING - ALL 65 tables)
        # ✅ Only runs if Oracle count succeeded
        if oracle_count is not None:
            try:
                oracle_min = max((oracle_count * (1.0 - ROW_COUNT_VARIANCE_PCT / 100.0)), DEFAULT_ROW_FLOOR)
                oracle_max = oracle_count * (1.0 + ROW_COUNT_VARIANCE_PCT / 100.0)
                safe_expect(v, cols, 'expect_table_row_count_to_be_between',
                           min_value=oracle_min, max_value=oracle_max, severity='warning')
            except Exception as e:
                print(f'   ⚠️  Row count check failed: {str(e)[:50]}')
        
        # CHECK 2: Null Check (WARNING - watermark + confirmed key columns)
        # ✅ Always runs (doesn't depend on Oracle)
        try:
            for col in null_check_cols:
                col_upper = col.strip().upper()
                if col_upper in cols:
                    safe_expect(v, cols, 'expect_column_values_to_not_be_null',
                               column=col_upper, severity='warning')
        except Exception as e:
            print(f'   ⚠️  Null check failed: {str(e)[:50]}')
        
        # CHECK 3: Uniqueness (WARNING - ONLY on 3 tables with confirmed keys)
        # ✅ Always runs (doesn't depend on Oracle)
        if has_uniqueness:
            try:
                for col in uniqueness_cols:
                    col_upper = col.strip().upper()
                    if col_upper in cols:
                        safe_expect(v, cols, 'expect_column_values_to_be_unique',
                                   column=col_upper, severity='warning')
            except Exception as e:
                print(f'   ⚠️  Uniqueness check failed: {str(e)[:50]}')
        
        # Save and run validation
        try:
            v.save_expectation_suite(discard_failed_expectations=False)
            run_and_score(suite, table_name, schema, br)
        except Exception as e:
            print(f'   ⚠️  Validation save/score failed: {str(e)[:50]}')
            failed_tables.append((i, table_name, f'Validation error: {str(e)[:50]}'))
    
    except Exception as e:  # ← OUTER CATCH: Any unexpected error
        error_msg = str(e)[:100]
        print(f' ✗ UNEXPECTED ERROR: {error_msg}')
        failed_tables.append((i, table_name, f'Unexpected error: {error_msg}'))
        continue

# Print summary
print(f'\n\n' + '='*80)
print(f'✅ VALIDATION COMPLETE FOR 65 TABLES')
print(f'='*80)

if failed_tables:
    print(f'\n⚠️  {len(failed_tables)} table(s) had issues:')
    for idx, tbl, reason in failed_tables:
        print(f'   [{idx:2d}/65] {tbl:40s} → {reason}')
else:
    print(f'\n✅ All 65 tables validated successfully!')

# ============================================================================
# END OF SECTION 5
# ============================================================================


[ 1/65] edw          . ABP_TAP                            

  argspec = oinspect.getargspec(obj)



 ✓ 693,139,354 rows

  warnings.warn(

  self.comm = Comm(**args)



   ⚠️  Oracle read failed for edw.ABP_TAP: (java.sql.SQLTimeoutException) ORA-01013: user requested cancel of current operation


JVM stacktrac


...

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

[PASS] raw.ncs_stage_STOP_POINT_TYPE: 1/1 ok (100%) warnings=0

[63/65] ncs_stage    . TRANSIT_FACILITY                    ✓      1,986 rows

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

[PASS] raw.ncs_stage_TRANSIT_FACILITY: 1/1 ok (100%) warnings=0

[64/65] edw          . USE_TRANSACTION_DAILY               ✗ UNEXPECTED ERROR: (java.lang.AssertionError) assertion failed: Conflicting directory structures detected. Suspicious p


✅ VALIDATION COMPLETE FOR 65 TABLES

⚠️  1 table(s) had issues:
   [64/65] USE_TRANSACTION_DAILY                    → Unexpected error: (java.lang.AssertionError) assertion failed: Conflicting directory structures detected. Suspicious p


## 6. Display results

In [0]:
print('\n--- Audit Table (Latest 300 records) ---')
display(spark.table(AUDIT_TABLE).orderBy('run_ts', ascending=False).limit(300))

print('\n--- Summary by Table ---')
display(spark.sql(f"""
SELECT table_name, schema_name, COUNT(*) as checks,
  SUM(CASE WHEN success THEN 1 ELSE 0 END) as passed,
  SUM(CASE WHEN NOT success THEN 1 ELSE 0 END) as failed_warnings
FROM {AUDIT_TABLE}
WHERE run_ts = (SELECT MAX(run_ts) FROM {AUDIT_TABLE})
GROUP BY table_name, schema_name
ORDER BY failed_warnings DESC, table_name
"""))


--- Audit Table (Latest 300 records) ---


table_name,schema_name,dq_check,column_name,success,severity,details,run_ts
TRANSIT_FACILITY,ncs_stage,expect_table_row_count_to_be_between,null,true,warning,1986,2026-08-27T15:28:44.925534Z
STOP_POINT_TYPE,ncs_stage,expect_table_row_count_to_be_between,null,true,warning,8,2026-08-27T15:28:19.002216Z
STOP_POINT_DIMENSION,edw,expect_table_row_count_to_be_between,null,true,warning,12888,2026-08-27T15:28:14.2066Z
STOP_POINT_DIMENSION,edw,expect_column_values_to_not_be_null,EDW_UPDATED_DTM,true,warning,expect_column_values_to_not_be_null(EDW_UPDATED_DTM): None,2026-08-27T15:28:14.2066Z
STOP_POINT,ncs_stage,expect_table_row_count_to_be_between,null,true,warning,12887,2026-08-27T15:28:06.866049Z
STOP_POINT,ncs_stage,expect_column_values_to_not_be_null,UPDATED_DTM,true,warning,expect_column_values_to_not_be_null(UPDATED_DTM): None,2026-08-27T15:28:06.866049Z
SLDC_MONTHLY_SUMMARY,cta,expect_table_row_count_to_be_between,null,true,warning,13376,2026-08-27T15:27:56.094253Z
SLDC_MONTHLY_SUMMARY,cta,expect_column_values_to_not_be_null,MONTH_DTM,true,warning,expect_column_values_to_not_be_null(MONTH_DTM): None,2026-08-27T15:27:56.094253Z
SERVICENOW_DATA_FROM_JUMPBOX,cta,expect_table_row_count_to_be_between,null,true,warning,604,2026-08-27T15:27:48.424348Z
SERVICENOW_DATA_FROM_JUMPBOX,cta,expect_column_values_to_not_be_null,U_START_DTM,true,warning,expect_column_values_to_not_be_null(U_START_DTM): None,2026-08-27T15:27:48.424348Z



--- Summary by Table ---


table_name,schema_name,checks,passed,failed_warnings
TRANSIT_FACILITY,ncs_stage,1,1,0


## 7. FINAL GATE (Always Passes — Status Report Mode)

In [0]:
tot_warn = sum(r['warning_count'] for r in RESULTS)

print(f"{'SUITE':<50} {'CHECKS':>7} {'PASSED':>7} {'WARNINGS':>8}")
print('-'*75)
for r in RESULTS:
    print(f"{r['suite']:<50} {r['evaluated']:>7} {r['successful']:>7} {r['warning_count']:>8}")
print('-'*75)
print(f"TOTAL warnings={tot_warn}\n")

print("📋 DQ CHECKS (All WARNING — Status Reporting Only):")
print("  • Row Count: WARNING on ALL 65 tables")
print("  • Null Check: WARNING on watermark/confirmed key columns (ALL 65 tables)")
print("  • Uniqueness: WARNING on 3 tables ONLY:")
print("    → availability_events (check: AVAILABILITY_EVENT_ID)")
print("    → device_current_hw_config (check: DEVICE_KEY, COMPONENT_SERIAL_NBR)")
print("    → device_event (check: DW_DEVICE_EVENT_ID)")
print("  • All other 62 tables: NO uniqueness check\n")

# STATUS REPORT: Show warnings but ALWAYS PASS
if tot_warn:
    print(f"⚠️  DQ STATUS REPORT: {tot_warn} warning(s) found\n")
    display(spark.table(AUDIT_TABLE)
            .filter("success = false")
            .orderBy('run_ts', ascending=False)
            .select('table_name', 'schema_name', 'dq_check', 'column_name', 'details', 'run_ts')
            .limit(100))
    print(f"\n✅ DQ GATE PASSED! (Warnings logged for status report)")
    print(f"   ✓ Row count: Status report available (65 tables)")
    print(f"   ✓ Null checks: Status report available (65 tables)")
    print(f"   ✓ Uniqueness: Status report available (3 tables)")
    print(f"   ✓ Warnings: {tot_warn} issue(s) logged (see audit table)")
    print(f"   ✓ Data is safe to promote")
else:
    print(f"✅ DQ GATE PASSED! (No issues found)")
    print(f"   ✓ Row count: ALL PASSED (65 tables)")
    print(f"   ✓ Null checks: ALL PASSED (65 tables)")
    print(f"   ✓ Uniqueness: ALL PASSED (3 tables)")
    print(f"   ✓ Data is safe to promote")

SUITE                                               CHECKS  PASSED WARNINGS
---------------------------------------------------------------------------
raw.edw_ABP_TAP                                          1       1        0
raw.cta_ABP_USE_TRAN_TIMING_DATA                         1       1        0
raw.edw_ABP_TAP                                          1       1        0
raw.cta_ABP_USE_TRAN_TIMING_DATA                         2       2        0
raw.ncs_stage_ACTIVITY_CODE                              1       1        0
raw.edw_ACTIVITY_CODE_DIMENSION                          1       1        0
raw.edw_AJ_METRIC_FACT                                   2       2        0
raw.edw_AVAILABILITY_EVENTS                              2       2        0
raw.cta_AVAILABILITY_LEVELS                              2       2        0
raw.edw_AVAILABILITY_PERIODS                             2       1        1
raw.edw_AVAILABILITY_RELIEF                              2       2        0
raw.ncs_stag

table_name,schema_name,dq_check,column_name,details,run_ts
SALE_TRANSACTION_DEVICE_MSG,ncs_stage,expect_table_row_count_to_be_between,null,366094,2026-08-27T15:27:24.577639Z
SALE_TRANSACTION,ncs_stage,expect_table_row_count_to_be_between,null,24610918,2026-08-27T15:25:48.114549Z
DEVICE_LAST_STATE,edw,expect_column_values_to_not_be_null,EVENT_DTM,expect_column_values_to_not_be_null(EVENT_DTM): None,2026-08-27T14:44:27.144086Z
DEVICE_END_OF_DAY_MSG_COUNT,ncs_stage,expect_table_row_count_to_be_between,null,42369945,2026-08-27T13:54:41.244852Z
DEVICE_END_OF_DAY,ncs_stage,expect_table_row_count_to_be_between,null,3901019,2026-08-27T13:50:21.951609Z
DEVICE_DIMENSION,edw,expect_column_values_to_not_be_null,INSERTED_DTM,expect_column_values_to_not_be_null(INSERTED_DTM): None,2026-08-27T13:47:35.899859Z
DEVICE,ncs_stage,expect_column_values_to_not_be_null,UPDATED_DTM,expect_column_values_to_not_be_null(UPDATED_DTM): None,2026-08-27T13:46:57.1013Z
AVAILABILITY_PERIODS,edw,expect_column_values_to_not_be_null,START_DTM,expect_column_values_to_not_be_null(START_DTM): None,2026-08-27T13:39:49.162401Z
SALE_TRANSACTION_DEVICE_MSG,ncs_stage,expect_table_row_count_to_be_between,null,366094,2026-08-27T12:48:11.238183Z
SALE_TRANSACTION,ncs_stage,expect_table_row_count_to_be_between,null,24610918,2026-08-27T12:45:54.932832Z



✅ DQ GATE PASSED! (Warnings logged for status report)
   ✓ Row count: Status report available (65 tables)
   ✓ Null checks: Status report available (65 tables)
   ✓ Uniqueness: Status report available (3 tables)
   ✓ Warnings: 23 issue(s) logged (see audit table)
   ✓ Data is safe to promote


## 8. Smoke test

In [0]:
def smoke_test():
    good=(1,'TVM',datetime.now(timezone.utc),5.0,2)
    bad=(1,'NOPE',None,-3.0,999)
    sdf=spark.createDataFrame([good,bad,(2,'GATE',datetime.now(timezone.utc),1.0,1)],
        'id int, cat string, ts timestamp, dur double, lvl int')
    cols={f.name for f in sdf.schema.fields}
    v,br=get_validator(sdf,'smoke.selftest')
    safe_expect(v,cols,'expect_column_values_to_not_be_null',column='ts',severity='warning')
    safe_expect(v,cols,'expect_column_values_to_be_in_set',column='cat',value_set=['TVM','GATE','VALIDATOR'],severity='warning')
    v.save_expectation_suite(discard_failed_expectations=False)
    r=run_and_score('smoke.selftest','<in-memory>','SMOKE',br)
    ok = r['warning_count'] >= 1  # Should have warnings
    print('\n' + ('='*75))
    print('SMOKE TEST', '✅ PASSED' if ok else '❌ FAILED')
    print('='*75)
    return ok

smoke_test()

  warnings.warn(

  self.comm = Comm(**args)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/19 [00:00<?, ?it/s]

[WARN] smoke.selftest: 0/2 ok (0%) warnings=2

SMOKE TEST ✅ PASSED


True